In [0]:
# Cell 1 — Cleanup before every run
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.bronze_transactions PURGE")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.silver_transactions PURGE")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.gold_sar_reports PURGE")

try:
    dbutils.fs.rm("/Volumes/aml_pipeline/transactions/raw_data/_checkpoints", recurse=True)
    print("Checkpoints cleared")
except:
    print("No checkpoints found - already clean")

tables = spark.sql("SHOW TABLES IN aml_pipeline.transactions").collect()
print(f"Tables remaining : {len(tables)}")
print("Cleanup complete - ready for fresh run")

Checkpoints cleared
Tables remaining : 0
Cleanup complete - ready for fresh run


In [0]:
# Cell 2 — Full pipeline: Bronze → Silver → Gold
# Single clean run. No duplicates. No assert.
# Checkpoint path: bronze_final (always cleared in Cell 1)

import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp, concat, date_format
from pyspark.sql.types import DoubleType, StringType, StructType, StructField, IntegerType

RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE    = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE      = "aml_pipeline.transactions.gold_sar_reports"
from datetime import datetime
RUN_ID          = datetime.now().strftime("%Y%m%d_%H%M%S")
CHECKPOINT_PATH = f"/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_{RUN_ID}"
print(f"Run ID          : {RUN_ID}")
print(f"Checkpoint path : {CHECKPOINT_PATH}")

# ── BRONZE ───────────────────────────────────────────────────
print("STEP 1: Bronze ingestion...")

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

(
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
).awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete : {bronze_count:,} records")

# ── SILVER ───────────────────────────────────────────────────
print("\nSTEP 2: Silver enrichment...")

try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback rates")

try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK           = {"KP","IR","MM","RU","BY","CU","SY","YE"}
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    if name.lower().strip() in OFAC:
        return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",   address),
        ("sender_name",      sender),
        ("sender_account",   s_acct),
        ("receiver_name",    receiver),
        ("receiver_account", r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
print(f"Silver complete : {silver_count:,} records")
print(f"  Flagged       : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"  Clean         : {silver_count-flagged_count:,} ({round((silver_count-flagged_count)/silver_count*100,1)}%)")

# ── GOLD ─────────────────────────────────────────────────────
print("\nSTEP 3: Gold SAR reports...")

(
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
    .write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
)

gold_count = spark.table(GOLD_TABLE).count()
critical   = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high       = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium     = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

print(f"\n{'='*50}")
print(f"  SENTINELFLOW PIPELINE COMPLETE")
print(f"{'='*50}")
print(f"  Bronze    : {bronze_count:,} raw transactions")
print(f"  Silver    : {silver_count:,} screened transactions")
print(f"  Gold      : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"  CRITICAL  : {critical:,}")
print(f"  HIGH      : {high:,}")
print(f"  MEDIUM    : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

Run ID          : 20260521_010122
Checkpoint path : /Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_20260521_010122
STEP 1: Bronze ingestion...
Bronze complete : 100,000 records

STEP 2: Silver enrichment...
Exchange rates fetched (30 currencies)
OFAC list loaded (470 names)
Silver complete : 100,000 records
  Flagged       : 3,356 (3.4%)
  Clean         : 96,644 (96.6%)

STEP 3: Gold SAR reports...

  SENTINELFLOW PIPELINE COMPLETE
  Bronze    : 100,000 raw transactions
  Silver    : 100,000 screened transactions
  Gold      : 3,356 SAR reports
  Flag rate : 3.4%
  CRITICAL  : 0
  HIGH      : 3,356
  MEDIUM    : 0

  Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|TRAVEL_RULE_VIOLATION: missing sender_address|1703 |
|HIGH_RISK_COUNTRY                            |1653 |
+---------------------------------------------+-----+



In [0]:
# Cell 3 — Final verification
b = spark.table("aml_pipeline.transactions.bronze_transactions").count()
s = spark.table("aml_pipeline.transactions.silver_transactions").count()
g = spark.table("aml_pipeline.transactions.gold_sar_reports").count()
print(f"Bronze : {b:,}")
print(f"Silver : {s:,}")
print(f"Gold   : {g:,}")
assert b == 100_000, f"Bronze count wrong: {b:,}"
assert s == 100_000, f"Silver count wrong: {s:,}"
print("All counts verified - pipeline is clean")

Bronze : 100,000
Silver : 100,000
Gold   : 3,356
All counts verified - pipeline is clean
